Assignment 10 by Chris

We will be using the **German Credit Data** dataset from the UCI Machine Learning Repository to analyze algorithmic fairness.

In [14]:
%pip install aif360

Setup Your Environment

Install necessary Python libraries

In [ ]:
%pip install numpy pandas scikit-learn aif360 BlackBoxAuditing

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 26.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for BlackBoxAuditing: filename=blackboxauditing-0.1.54-py2.py3-none-any.whl size=1394818 sha256=28badd3c87e253afc61829569433a95421459bd0f444cf862d1c49af756779f0
  Stored in directory: /root/.cache/pip/wheels/4a/96/d6/0176976ee1785bb3d685ea52543b319b0a994fa3b38130ab96
Successfully built BlackBoxAuditing


Load and Preprocess the Data

Import libraries and load the data

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from aif360.datasets import StandardDataset

# Load German Credit Data
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
column_names = [
    'status', 'duration', 'credit_history', 'purpose', 'amount',
    'savings', 'employment_duration', 'installment_rate',
    'personal_status_sex', 'other_debtors', 'present_residence',
    'property', 'age', 'other_installment_plans', 'housing',
    'number_credits', 'job', 'people_liable', 'telephone',
    'foreign_worker', 'credit_risk'
]

df = pd.read_csv(url, sep=' ', names=column_names)

# Map credit_risk: 1 = Good (Favorable), 2 = Bad (Unfavorable) -> Map to 1 and 0
df['credit_risk'] = df['credit_risk'].map({1: 1, 2: 0})

# Decode 'personal_status_sex' to extract 'sex' as a binary attribute
df['sex'] = df['personal_status_sex'].map({
    'A91': 'male', 'A93': 'male', 'A94': 'male',
    'A92': 'female', 'A95': 'female'
})

# Identify categorical features to encode
categorical_features = [
    'status', 'credit_history', 'purpose', 'savings',
    'employment_duration', 'personal_status_sex', 'other_debtors',
    'property', 'other_installment_plans', 'housing', 'job',
    'telephone', 'foreign_worker', 'sex'
]

# Encode categorical variables
encoders = {}
for feature in categorical_features:
    encoders[feature] = LabelEncoder()
    df[feature] = encoders[feature].fit_transform(df[feature].astype(str))

# Create binary age attribute (e.g., age > 25 as 1, <=25 as 0)
df['age_cat'] = df['age'].apply(lambda x: 1 if x > 25 else 0)

# Get encoded values for 'male'
male_encoded = encoders['sex'].transform(['male'])[0]

# Define 'sex' and 'age_cat' as protected attributes
data = StandardDataset(
    df,
    label_name='credit_risk',
    favorable_classes=[1],
    protected_attribute_names=['sex', 'age_cat'],
    privileged_classes=[[male_encoded], [1]]
)

print("Dataset successfully loaded and prepared!")

Dataset successfully loaded and prepared!


Split the Data And Train using traditional approach

Divide the data into training and testing sets.

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from aif360.metrics import BinaryLabelDatasetMetric

# Split into Train and Test
train, test = data.split([0.7], shuffle=True, seed=42)

# Unprivileged groups: sex = female (0), age_cat = young (0)
# Privileged groups: sex = male (1), age_cat = old (1)
unprivileged_groups = [{'sex': 0, 'age_cat': 0}]
privileged_groups = [{'sex': 1, 'age_cat': 1}]

# Train Baseline Logistic Regression Model
model_baseline = LogisticRegression(solver='liblinear', random_state=42)
model_baseline.fit(train.features, train.labels.flatten())

# Predict and evaluate
predictions_baseline = model_baseline.predict(test.features)
accuracy_baseline = accuracy_score(test.labels.flatten(), predictions_baseline)

test_pred = test.copy()
test_pred.labels = predictions_baseline.reshape(-1, 1)

metric_baseline = BinaryLabelDatasetMetric(
    test_pred,
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

disparity_baseline = metric_baseline.disparate_impact()
print("Baseline Accuracy:", accuracy_baseline)
print("Baseline Disparate Impact:", disparity_baseline)

pip install 'aif360[Reductions]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[Reductions]'


Baseline Accuracy: 0.8033333333333333
Baseline Disparate Impact: 0.8214285714285715


Apply a Fairness Mitigation Technique

In [11]:
from aif360.algorithms.preprocessing import Reweighing

# Apply Reweighing (First Intervention)
RW = Reweighing(
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)
train_rw = RW.fit_transform(train)
print("Reweighing weights computed successfully!")

Reweighing weights computed successfully!


Train a Model Using the Reweighted Data

Train a logistic regression model using the transformed data.

In [12]:
# Train a Logistic Regression model with computed sample weights
model_rw = LogisticRegression(solver='liblinear', random_state=42)
model_rw.fit(train_rw.features, train_rw.labels.flatten(), sample_weight=train_rw.instance_weights.flatten())

predictions_rw = model_rw.predict(test.features)
accuracy_rw = accuracy_score(test.labels.flatten(), predictions_rw)

test_pred_rw = test.copy()
test_pred_rw.labels = predictions_rw.reshape(-1, 1)

metric_rw = BinaryLabelDatasetMetric(
    test_pred_rw,
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

disparity_rw = metric_rw.disparate_impact()
print("Reweighted Accuracy:", accuracy_rw)
print("Reweighted Disparate Impact:", disparity_rw)

Reweighted Accuracy: 0.7833333333333333
Reweighted Disparate Impact: 1.0661764705882353


Evaluate the Model

Evaluate the fairness and accuracy of the model on the test set

In [15]:
try:
    import BlackBoxAuditing
except ImportError:
    !pip install BlackBoxAuditing

from aif360.algorithms.preprocessing import DisparateImpactRemover
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from aif360.metrics import BinaryLabelDatasetMetric

# Apply Disparate Impact Remover (Second Intervention)
DIR = DisparateImpactRemover(repair_level=1.0, sensitive_attribute='sex')
train_dir = DIR.fit_transform(train)
test_dir = DIR.fit_transform(test)

model_dir = LogisticRegression(solver='liblinear', random_state=42)
model_dir.fit(train_dir.features, train_dir.labels.flatten())

predictions_dir = model_dir.predict(test_dir.features)
accuracy_dir = accuracy_score(test_dir.labels.flatten(), predictions_dir)

test_pred_dir = test_dir.copy()
test_pred_dir.labels = predictions_dir.reshape(-1, 1)

metric_dir = BinaryLabelDatasetMetric(
    test_pred_dir,
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

disparity_dir = metric_dir.disparate_impact()
print("Disparate Impact Remover Accuracy:", accuracy_dir)
print("Disparate Impact Remover Disparate Impact:", disparity_dir)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 17.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for BlackBoxAuditing: filename=blackboxauditing-0.1.54-py2.py3-none-any.whl size=1394818 sha256=d7516f03eaa12f1ad37a678247de91cfba043424732638a236c61ce11345e03c
  Stored in directory: /root/.cache/pip/wheels/4a/96/d6/0176976ee1785bb3d685ea52543b319b0a994fa3b38130ab96
Successfully built BlackBoxAuditing
Disparate Impact Remover Accuracy: 0.8
Disparate Impact Remover Disparate Impact: 0.7876712328767124


### Visualizing Results
We will compare the **Accuracy** and **Disparate Impact** metrics of our three configurations:
1. **Baseline Model** (without any mitigation)
2. **Reweighing** (Pre-processing intervention)
3. **Disparate Impact Remover** (Pre-processing intervention)

In [16]:
import matplotlib.pyplot as plt

models = ['Baseline', 'Reweighing', 'Disparate Impact Remover']
accuracies = [accuracy_baseline, accuracy_rw, accuracy_dir]
disparities = [disparity_baseline, disparity_rw, disparity_dir]

fig, ax1 = plt.subplots(figsize=(8, 5))

# Plot Accuracy
color = 'tab:blue'
ax1.set_xlabel('Model Configuration', fontweight='bold')
ax1.set_ylabel('Accuracy', color=color, fontweight='bold')
bars = ax1.bar(models, accuracies, color=color, alpha=0.6, width=0.4, label='Accuracy')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(0, 1.0)

# Line for 80% rule in disparate impact (ideal is 1.0)
ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Disparate Impact', color=color, fontweight='bold')
line = ax2.plot(models, disparities, color=color, marker='o', linewidth=2, label='Disparate Impact')
ax2.tick_params(axis='y', labelcolor=color)
ax2.axhline(0.8, color='green', linestyle='--', alpha=0.7, label='80% Fairness Threshold')
ax2.axhline(1.0, color='gold', linestyle='-', alpha=0.5, label='Ideal Equality (1.0)')

plt.title('Comparison of Accuracy vs. Disparate Impact', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

In [21]:
import numpy as np
import matplotlib.pyplot as plt

# Données à afficher
model_names = ['Baseline', 'Reweighing', 'Disparate Impact Remover']
accuracy_vals = [accuracy_baseline, accuracy_rw, accuracy_dir]
disparity_vals = [disparity_baseline, disparity_rw, disparity_dir]

x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

# Génération des barres pour la Précision et l'Impact Disparate
rects1 = ax.bar(x - width/2, accuracy_vals, width, label='Précision (Accuracy)', color='#3498db')
rects2 = ax.bar(x + width/2, disparity_vals, width, label='Impact Disparate (Disparate Impact)', color='#e74c3c')

# Ajout des détails et légendes
ax.set_ylabel('Scores / Valeurs', fontsize=12, fontweight='bold')
ax.set_title('Comparaison de la Précision et de l\'Impact Disparate par Modèle', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=11)
ax.legend(loc='upper right')

# Ligne de référence pour le seuil d'équité de 80% (0.8)
ax.axhline(0.8, color='green', linestyle='--', alpha=0.7, label='Seuil d\'équité de 80%')

# Affichage des valeurs exactes au-dessus des barres
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # Décalage vertical de 3 points
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.show()